In [ ]:
# === SETUP ===
import os, sys, yaml, json, subprocess, shutil, urllib.request, zipfile, glob
os.chdir('/kaggle/working')

# Copy kit files from Dataset to working
kit_srcs = glob.glob('/kaggle/input/*')
kit_src = next((s for s in kit_srcs if os.path.isdir(os.path.join(s, 'src'))), None)
if kit_src:
    for item in os.listdir(kit_src):
        if item.startswith('.'): continue
        s = os.path.join(kit_src, item)
        d = os.path.join('/kaggle/working', item)
        if os.path.isdir(s):
            shutil.copytree(s, d, dirs_exist_ok=True)
        else:
            shutil.copy2(s, d)
print('Kit root:', os.getcwd())


In [ ]:
# === DOWNLOAD WAN2.2-S2V Q8 GGUF ===
cfg = yaml.safe_load(open('config.yaml'))
models_dir = cfg['paths']['models']
os.makedirs(models_dir, exist_ok=True)

model_url = 'https://huggingface.co/Kijai/Wan2.1-GGUF/resolve/main/wan2.1-s2v-Q8_0.gguf'
model_path = os.path.join(models_dir, 'wan2.1-s2v-Q8_0.gguf')

if not os.path.exists(model_path):
    print('Downloading Wan2.2-S2V Q8 GGUF (~14GB)...')
    urllib.request.urlretrieve(model_url, model_path)
    print(f'Done: {os.path.getsize(model_path)/1e9:.1f} GB')
else:
    print(f'Cached: {os.path.getsize(model_path)/1e9:.1f} GB')

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['KIT_CONFIG'] = 'config.yaml'
cfg['active_profile'] = 'kaggle_2xt4'
yaml.safe_dump(cfg, open('config.yaml', 'w'), sort_keys=False)
print('Profile: kaggle_2xt4 (FREE, 2x T4)')


In [ ]:
# 2) Confirm the free 2xT4 profile
import yaml
c=yaml.safe_load(open('config_lastlaugh.yaml')); c['active_profile']='kaggle_2xt4_max'
yaml.safe_dump(c, open('config_lastlaugh.yaml','w'), sort_keys=False)
print('active_profile ->', c['active_profile'])


In [ ]:
# 3) FREE validation (no GPU spend)
!KIT_CONFIG=config_lastlaugh.yaml python3 src/selftest.py --quick
!KIT_CONFIG=config_lastlaugh.yaml python3 src/preflight.py
!COMFY_MOCK=1 KIT_CONFIG=config_lastlaugh.yaml python3 src/render_stills.py --smoke


In [ ]:
# 4) REAL orchestral score on the T4 (free)
!KIT_CONFIG=config_lastlaugh.yaml python3 src/score_gen.py --root .


In [ ]:
# 5) PASS 1 - lock a pixel-perfect STILL per hero beat (best face control, cheap)
for S in ['gift_firstlight','mask_becoming','confr_real','confr_smile']:
    print('===== STILL', S, '=====')
    os.system(f'KIT_CONFIG=config_lastlaugh.yaml python3 src/render_stills.py --only {S}')


In [ ]:
# 6) DIGNITY GATE #1 - review the locked stills BEFORE any motion
work='/kaggle/working/lastlaugh_work'
stills=[f'{work}/{S}/still.png' for S in ['gift_firstlight','mask_becoming','confr_real','confr_smile']]
stills=[p for p in stills if os.path.exists(p)]
if stills:
    n=len(stills); ins=' '.join('-i "%s"'%p for p in stills)
    fc=';'.join('[%d]scale=-1:360[s%d]'%(i,i) for i in range(n))+';'+''.join('[s%d]'%i for i in range(n))+'hstack=inputs=%d[o]'%n
    os.system('ffmpeg -y %s -filter_complex "%s" -map "[o]" /kaggle/working/STILLS_REVIEW.png'%(ins,fc))
    print('Review -> /kaggle/working/STILLS_REVIEW.png  (open it; approve the FACES before animating)')
else:
    print('No stills found - check PASS 1 output above.')


In [ ]:
# 7) PASS 2 - ANIMATE each locked still into REAL motion.
# Drop the freeze mp4 (KEEP still.png) so the motion pass actually runs instead of skipping.
for S in ['gift_firstlight','mask_becoming','confr_real','confr_smile']:
    fp=f'/kaggle/working/lastlaugh_work/{S}/final.mp4'
    if os.path.exists(fp): os.remove(fp)
    print('===== ANIMATE', S, '=====')
    os.system(f'KIT_CONFIG=config_lastlaugh.yaml KIT_MOTION_DENOISE=0.45 python3 src/render_motion.py --only {S}')


In [ ]:
# 8) Preview reel of the 4 MOTION clips (judge motion + face; dignity gate #2)
work='/kaggle/working/lastlaugh_work'; beats=['gift_firstlight','mask_becoming','confr_real','confr_smile']
open(work+'/preview_list.txt','w').write(''.join("file '%s/final.mp4'\n" % b for b in beats))
os.system('ffmpeg -y -f concat -safe 0 -i %s/preview_list.txt -c copy /kaggle/working/PREVIEW_4BEATS_MOTION.mp4' % work)
print('Preview -> /kaggle/working/PREVIEW_4BEATS_MOTION.mp4  (download from Output tab, watch full-screen)')


In [ ]:
# 9) FULL FILM (only after the 4 test beats satisfy you). Same two-pass, all 74 beats, resumable.
# STEP A - stills for every beat (rerun each session to resume):
# os.system('KIT_CONFIG=config_lastlaugh.yaml python3 src/render_stills.py --max-hours 8')
# STEP B - REVIEW all stills under /kaggle/working/lastlaugh_work/*/still.png
# STEP C - drop freeze mp4s (KEEP stills) so the motion pass will run:
# os.system('find /kaggle/working/lastlaugh_work -name final.mp4 -delete')
# STEP D - animate every beat into motion (rerun each session to resume):
# os.system('KIT_CONFIG=config_lastlaugh.yaml KIT_MOTION_DENOISE=0.45 python3 src/render_motion.py --max-hours 8')
# STEP E - assemble for free (CPU):
# os.system('POST_MODE=cpu KIT_CONFIG=config_lastlaugh.yaml bash src/assemble.sh')
# os.system('cd output_lastlaugh && zip -qr /kaggle/working/THE_LAST_LAUGH_final.zip THE_LAST_LAUGH.mp4')


In [ ]:
# === QUALITY GATE + PACKAGE ===
!python3 src/qc.py

out_dir, targets = 'output', ['THE_MOMENT.mp4', 'THE_MOMENT_MASTER_4444.mov']
existing = [f for f in targets if os.path.exists(os.path.join(out_dir, f))]
zip_path = '/kaggle/working/THE_MOMENT_final.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in existing:
        z.write(os.path.join(out_dir, f), f)
print(f'Delivery zip: {zip_path}')
print('Download from Kaggle Output tab.')
